# Single-chain model validation

In [ ]:
from functools import partial
import sys
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / "analysis" / "src" / "plots" / "style.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
plt.rcParams.update(STYLE.mpl_rc())
OUTPUT_ROOT = ROOT / "analysis_results"

In [ ]:
from src.plots.figures import (
    plot_mfe_pairing_scatter, plot_precision_recall, plot_rg_validation,
)

SWEEP = "53c_single_mol_duplex"
DENSITY = None
RG_REFERENCE_NM = 1.47

replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
if DENSITY is not None:
    replicates = filter_rows(replicates, density=DENSITY)
    summary = filter_rows(summary, density=DENSITY)
E_VALUES = sorted(summary["E_att"].dropna().unique())

## MFE-matching versus additional base pairs

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(plot_mfe_pairing_scatter, style=STYLE, attraction_order=E_VALUES),
    style=STYLE, ncols=3, width=4, height=3.5, legend="bottom",
)
for ax, stiffness in zip(axes.flat, sorted(replicates["angle_k_ppp"].unique())):
    ax.set_title(rf"$k_{{\mathrm{{bbb}}}}={stiffness:.1f}$")
    ax.set_xlim(-.25, 12.6)
    ax.set_ylim(-.25, 10.7 if stiffness == 0 else 4.6)
fig.legends[0].set_title(r"$E_{\mathrm{att}}$")
plt.show()

## Precision and recall

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(plot_precision_recall, style=STYLE, attraction_order=E_VALUES),
    style=STYLE, ncols=3, width=3.9, height=3.5,
    sharex=True, sharey=True, legend="last",
)
for ax, stiffness in zip(axes.flat, sorted(replicates["angle_k_ppp"].unique())):
    ax.set_title(rf"$k_{{\mathrm{{bbb}}}}={stiffness:.1f}$")
    ax.set_aspect("equal", adjustable="box")
axes.flat[-1].get_legend().set_title(r"$E_{\mathrm{att}}$")
plt.show()

## Radius of gyration

In [ ]:
from src.plots.data import load_rg_frames

rg_replicates, rg_summary = load_rg_frames(
    OUTPUT_ROOT, SWEEP,
    late_fraction=.20,
    density=DENSITY,
    input_sweep=ROOT.parent / "validation" / "bulk" / "output_dir" / SWEEP,
)
fig, axes = panels_by(
    rg_summary, by="angle_k_ppp",
    plot_fn=lambda d, ax: plot_rg_validation(
        d, ax,
        replicates=filter_rows(
            rg_replicates, angle_k_ppp=d["angle_k_ppp"].iloc[0],
        ),
        style=STYLE, x_order=E_VALUES, reference_nm=RG_REFERENCE_NM,
    ),
    style=STYLE, ncols=3, width=4, height=3.4,
    sharey=True, legend="none",
)
for ax, stiffness in zip(axes.flat, sorted(rg_summary["angle_k_ppp"].unique())):
    ax.set_title(rf"$k_{{\mathrm{{bbb}}}}={stiffness:.1f}$")
    ax.set_ylim(.5, 3.6)
    ax.set_yticks([.5, 1, 1.5, 2, 2.5, 3, 3.5])
plt.show()